# Standard Single-Objective Acquisition Functions

同じ`SingleTaskGP` posteriorから、代表的なstandard BO acquisitionが**何を価値とみなし、どのcandidateを返すか**を比較します。

関連理論:

- [Improvement](../../docs/theory/acquisition/02_improvement.md)
- [Confidence Bound](../../docs/theory/acquisition/03_confidence_bound.md)
- [Lookahead](../../docs/theory/acquisition/06_lookahead.md)
- [Selection Guide](../../docs/theory/acquisition/12_selection_guide.md)

robotorchanは標準acquisitionを再実装せず、BoTorch native classesを直接利用します。

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.analytic import (
    LogExpectedImprovement,
    LogProbabilityOfImprovement,
    UpperConfidenceBound,
)
from botorch.acquisition.knowledge_gradient import qKnowledgeGradient
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.monte_carlo import qSimpleRegret, qUpperConfidenceBound
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Problem and surrogate

最大化問題として統一します。noise-freeの簡潔な比較なのでimprovement familyの基準は観測`train_Y.max()`を使います。

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.35 * torch.cos(5 * torch.pi * X)


train_X = torch.tensor([[0.05], [0.18], [0.38], [0.58], [0.78], [0.95]])
train_Y = objective(train_X)
bounds = torch.tensor([[0.0], [1.0]])

model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()
print("best observed value:", train_Y.max())

## 3. Same posterior, different utilities

acquisition familyを変えてもsurrogate posteriorは同じです。違うのはposteriorをdecision utilityへ変換する規則です。

In [ ]:
grid = torch.linspace(0.0, 1.0, 300).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
mean = posterior.mean.squeeze(-1)
std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid.squeeze(-1), mean, label="posterior mean")
ax.fill_between(grid.squeeze(-1), mean - 2 * std, mean + 2 * std, alpha=0.2)
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")
ax.legend()
plt.show()

## 4. Analytic q=1: LogEI, LogPI, UCB

- LogEI: improvement量の期待値をlog-domainで安定に扱います。
- LogPI: improvement eventの確率を重視します。
- UCB: posterior meanとuncertainty bonusを直接組み合わせます。

Log formulationは別の探索原理ではありません。

In [ ]:
analytic_acqfs = {
    "LogEI": LogExpectedImprovement(model=model, best_f=train_Y.max()),
    "LogPI": LogProbabilityOfImprovement(model=model, best_f=train_Y.max()),
    "UCB": UpperConfidenceBound(model=model, beta=0.2),
}

analytic_candidates = {}
for name, acqf in analytic_acqfs.items():
    candidate, value = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=1,
        num_restarts=5,
        raw_samples=64,
    )
    analytic_candidates[name] = candidate
    print(name, "candidate=", candidate.item(), "value=", value.item())

## 5. MC q=1: qLogEI, qUCB, qSimpleRegret

q=1でもMC formulationを使えます。`qSimpleRegret`はposterior utilityとして候補値そのものを重視し、improvement thresholdを必要としません。

In [ ]:
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([128]), seed=123)
mc_acqfs = {
    "qLogEI": qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
        sampler=sampler,
    ),
    "qUCB": qUpperConfidenceBound(model=model, beta=0.2, sampler=sampler),
    "qSimpleRegret": qSimpleRegret(model=model, sampler=sampler),
}

mc_candidates = {}
for name, acqf in mc_acqfs.items():
    candidate, value = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=1,
        num_restarts=5,
        raw_samples=64,
    )
    mc_candidates[name] = candidate
    print(name, "candidate=", candidate.item(), "value=", value.item())

## 6. Candidate locations are the decision output

acquisition valueの絶対値をfamily間で比較して優劣を付けません。utilityの定義・scaleが違うためです。比較対象は各criterionがどの意思決定を表すかと、結果としてどこをcandidateにするかです。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), objective(grid).squeeze(-1), label="objective")
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observations")

all_candidates = {**analytic_candidates, **mc_candidates}
for name, candidate in all_candidates.items():
    y = objective(candidate).item()
    ax.scatter(candidate.item(), y, marker="x", s=70, label=name)

ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.legend(ncol=2)
plt.show()

## 7. Knowledge Gradient is structurally different

KGはcandidate自身のimprovementではなく、観測後のterminal decision valueを評価します。`qKnowledgeGradient`はone-shot acquisitionなので、optimizer内部ではreal candidateに加えてauxiliary fantasy decision rowsも最適化します。

In [ ]:
kg = qKnowledgeGradient(
    model=model,
    num_fantasies=4,
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([4]), seed=321),
)
augmented_q = kg.get_augmented_q_batch_size(q=1)
print("real q:", 1)
print("augmented q:", augmented_q)

## 8. qKG candidate generation and extraction

`optimize_acqf`はone-shot acquisitionの`extract_candidates` semanticsを認識し、返却値は実際に評価するcandidateだけになります。auxiliary rowsを実験点として扱いません。

In [ ]:
kg_candidate, kg_value = optimize_acqf(
    acq_function=kg,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)

assert kg_candidate.shape == torch.Size([1, 1])
print("qKG actual candidate:", kg_candidate)
print("qKG value:", kg_value)

## 9. One candidate evaluation

ここでは代表としてqLogEI candidateを評価します。acquisition比較のNotebookなので、各手法を順番に評価してtraining dataを変えることはしません。

In [ ]:
candidate = mc_candidates["qLogEI"]
candidate_Y = objective(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)
print("candidate:", candidate)
print("evaluation:", candidate_Y)
print("updated shapes:", updated_X.shape, updated_Y.shape)

## 10. What changes for q > 1

`qLogEI`や`qUCB`の`q>1`はpointwise score上位q点ではありません。joint posterior samplesを使ってcandidate set全体を価値付けします。

```text
q=1 analytic
    marginal posterior momentsで閉形式を使える場合がある

q>1 MC
    joint posterior / candidate dependenceが重要

qKG
    q real candidates + auxiliary fantasy decision rows
```

Batch / pending / noisy semanticsは専用Phaseで扱います。

## 11. Selection semantics

| Family | Main decision quantity | Typical question |
| --- | --- | --- |
| LogEI / qLogEI | expected improvement | incumbentをどれだけ改善できるか |
| LogPI | probability of improvement | 改善eventが起きる確率は高いか |
| UCB / qUCB | optimistic posterior utility | meanとuncertaintyをどうtrade-offするか |
| qSimpleRegret | posterior utility of selected points | 良い最終点を選ぶにはどこか |
| qKG | value of information | 観測後のterminal decisionがどれだけ改善するか |

acquisition valueそのものを異なるfamily間で横並び比較しません。

## 12. Important boundaries

- noiseが強い場合は単純なobserved `best_f`よりqLogNEI等を検討します。
- q-batchとsequential top-q rankingを同一視しません。
- `X_pending`とKGのfantasy rowsは別物です。
- outcome constraintとcandidate/input-space constraintは別レイヤーです。
- multi-objective、multi-fidelity、active learningは別のdecision purposeです。
- standard acquisitionはBoTorch native pathを優先し、robotorchan wrapperを増やしません。

## 13. References

数式、posterior requirement、q/noisy/lookaheadの詳細は冒頭のTheory章を参照してください。APIの現在の推奨は`docs/optimization/standard_acquisition.md`にあります。